In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%pylab inline
import scipy.io

import os
import pandas as pd

In [ ]:
features_df = pd.read_csv('/content/drive/MyDrive/features_df.csv') #You might have to change this path

In [ ]:
#RUN THIS ONLY IF YOU CANNOT load features_df

import numpy as np
from scipy.stats import skew, kurtosis
from scipy.signal import welch
import pandas as pd

# Assuming calculate_shannons_entropy and calculate_hjorth_parameters functions are defined

def calculate_features(segment, fs=128):
    features = []
    # Assuming segment shape is (samples, channels)
    for channel_index in range(segment.shape[1]):
        channel_data = segment[:, channel_index]

        # Calculate features for each channel
        std_dev = np.std(channel_data)
        rms = np.sqrt(np.mean(channel_data**2))
        skewness = skew(channel_data)
        kurt = kurtosis(channel_data)
        # PSD using Welch's method
        freqs, psd = welch(channel_data, fs=fs)
        spectral_entropy = -np.sum(psd / np.sum(psd) * np.log2(psd / np.sum(psd)))
        # Calculate band power using the PSD
        # This step is already based on the band-specific segment
        band_power = np.sum(psd)

        # Hjorth parameters
        activity, mobility, complexity = calculate_hjorth_parameters(channel_data)
        shannons_entropy = calculate_shannons_entropy(channel_data)

        # Compile features for the current channel
        features.append({
            'std_dev': std_dev,
            'rms': rms,
            'skewness': skewness,
            'kurtosis': kurt,
            'spectral_entropy': spectral_entropy,
            'band_power': band_power,
            'hjorth_activity': activity,
            'hjorth_mobility': mobility,
            'hjorth_complexity': complexity,
            'shannons_entropy': shannons_entropy
        })
    return features

# Initialize a DataFrame to store all features
all_features_list = []

for band in segments_by_band_with_meta.keys():
    for segment_meta in segments_by_band_with_meta[band]:
        segment = segment_meta[0]
        file_id = segment_meta[1]
        group = segment_meta[2]
        print(f"Processing file: {file_id}, band: {band}")
        segment_features = calculate_features(segment)
        for channel_index, channel_features in enumerate(segment_features):
            # Add segment, band, channel info, and metadata
            channel_features.update({
                'band': band,
                'file_id': file_id,
                'group': group,
                'channel_id': channel_index
            })
            all_features_list.append(channel_features)

# Create a DataFrame from the list of feature dictionaries
features_df = pd.DataFrame(all_features_list)

# Now features_df contains the features for all segments, bands, and channels, along with metadata
print(features_df.head())

features_df.to_csv('features_df.csv', index=False)


In [ ]:
pip install statsmodels

In [ ]:
features_df.head()

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
import pandas as pd

# Assuming features_df is your DataFrame and it includes a 'group' column
# Replace 'feature_name' with the actual column name for a feature you want to test

# Perform ANOVA for each feature of interest
features_to_test = ['std_dev', 'rms', 'skewness', 'kurtosis', 'spectral_entropy', 'band_power', 'hjorth_activity', 'hjorth_mobility', 'hjorth_complexity', 'shannons_entropy']
anova_results = []

for feature in features_to_test:
    model = ols(f'{feature} ~ C(group)', data=features_df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)  # typ=2 for Type II SS
    p_value = anova_table['PR(>F)'][0]
    anova_results.append({
        'Feature': feature,
        'F': anova_table['F'][0],
        'p-value': p_value,
    })

anova_results_df = pd.DataFrame(anova_results)
anova_results_df.sort_values(by='p-value', inplace=True)
print(anova_results_df)


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Assuming features_df is your DataFrame from the previous step
# Drop non-feature columns
feature_columns = [col for col in features_df.columns if col not in ['band', 'file_id', 'group', 'channel_id']]
features = features_df[feature_columns]

# Standardize the features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Perform PCA
pca = PCA()
components = pca.fit_transform(features_scaled)

print(pca.n_components_)

# Create a DataFrame with the PCA components
loadings = pca.components_
loadings_df = pd.DataFrame(loadings.T, columns=[f'PC{i+1}' for i in range(loadings.shape[0])], index=feature_columns)

# Explained variance can help determine how many components to consider
explained_variance = pca.explained_variance_ratio_
cumulative_explained_variance = explained_variance.cumsum()

# Identify important features based on loadings
# For simplicity, we're looking at PC1 here, but you should consider other components too
important_features_pc1 = loadings_df['PC1'].abs().sort_values(ascending=False)

print("Important features for PC1:", important_features_pc1.head())
print("Cumulative explained variance:", cumulative_explained_variance)


In [ ]:
from sklearn.model_selection import train_test_split

# Assuming 'features_df' contains all your features and 'group' is your target variable
X = features_df.drop(['group', 'file_id', 'band', 'channel_id'], axis=1)  # Drop non-feature columns
y = features_df['group']  # Target variable

# First split to separate out the test set
X_intermediate, X_test, y_intermediate, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42)

# Now split the intermediate set into training and validation sets
X_train, X_validation, y_train, y_validation = train_test_split(
    X_intermediate, y_intermediate, test_size=0.1765, random_state=42)  # 0.1765 * (1 - 0.15) ≈ 0.15

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_validation.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


In [ ]:
#THIS IS THE CODE THAT RUNS FOREVER - most likely a bug

from sklearn.model_selection import KFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

# Assuming X is your features and y is your target variable
X = np.array(features_df.drop(['group', 'file_id', 'channel_id', 'band'], axis=1))
y = np.array(features_df['group'])

# Define the SVM classifier
svm_classifier = SVC(kernel='rbf')  # Gaussian kernel

# Define the k-fold cross-validation setup
kf = KFold(n_splits=10, shuffle=True, random_state=42)

fold = 0
for train_index, test_index in kf.split(X):
    print(f"TRAIN indices:", train_index)
    print(f"TEST indices:", test_index)
    fold += 1
    print(f"Starting fold {fold}")

    # Split the data into training and test sets for the current fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train the model
    svm_classifier.fit(X_train, y_train)

    # Predict on the test set
    y_pred = svm_classifier.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Fold {fold}: Accuracy = {accuracy}")

    # You can add any additional metrics or operations you need here


In [ ]:
# Train the classifier on the training set
svm_clf.fit(X_train, y_train)

# Validate the model
validation_predictions = svm_clf.predict(X_validation)
validation_accuracy = accuracy_score(y_validation, validation_predictions)
print(f"Validation Accuracy: {validation_accuracy}")

# Test the model
test_predictions = svm_clf.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Test Accuracy: {test_accuracy}")